## Installation

In [ ]:
!pip install nnunetv2 -q

## Imports and Environment Variables


In [ ]:
import os
import json
import glob
import random
import shutil
import subprocess
import numpy as np
import nibabel as nib
from tqdm.auto import tqdm

# Environment variables — must be set at every session start
os.environ["nnUNet_raw"]          = "/kaggle/working/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/kaggle/working/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = "/kaggle/working/nnUNet_results"
os.environ["nnUNet_n_proc_DA"]    = "2"

# Create directories
for path in [os.environ["nnUNet_raw"],
             os.environ["nnUNet_preprocessed"],
             os.environ["nnUNet_results"]]:
    os.makedirs(path, exist_ok=True)

print("Environment variables set:")
for k in ["nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"]:
    print(f"  {k} = {os.environ[k]}")

## Find Dataset Files


In [ ]:
BASE1 = "/kaggle/input/datasets/andrewmvd/liver-tumor-segmentation"
BASE2 = "/kaggle/input/datasets/andrewmvd/liver-tumor-segmentation-part-2"

vol_files = (
    glob.glob(f"{BASE1}/volume_pt*/*.nii") +
    glob.glob(f"{BASE2}/volume_pt*/*.nii") +
    glob.glob(f"{BASE1}/volume_pt*/*.nii.gz") +
    glob.glob(f"{BASE2}/volume_pt*/*.nii.gz")
)

seg_files = (
    glob.glob(f"{BASE1}/segmentations/*.nii") +
    glob.glob(f"{BASE1}/segmentations/*.nii.gz")
)

print(f"Volume count      : {len(vol_files)}")
print(f"Segmentation count: {len(seg_files)}")

print("\nVolume examples:")
for f in sorted(vol_files)[:3]:
    print(f"  {f}")

print("\nSegmentation examples:")
for f in sorted(seg_files)[:3]:
    print(f"  {f}")

## Case ID Map and Train/Val Split

In [ ]:
def get_case_id(path):
    name = os.path.basename(path)
    digits = ''.join(filter(str.isdigit, name))
    return int(digits)

# ID → path map
vol_map = {get_case_id(p): p for p in vol_files}
seg_map = {get_case_id(p): p for p in seg_files}

# Common IDs
all_ids = sorted(set(vol_map.keys()) & set(seg_map.keys()))
print(f"Matched cases: {len(all_ids)}")

# Train / Val split — fixed seed
random.seed(42)
shuffled = all_ids.copy()
random.shuffle(shuffled)

VAL_IDS   = sorted(shuffled[111:])   # 20 validation cases
TRAIN_IDS = sorted(shuffled[:111])   # 111 training cases

print(f"Train cases: {len(TRAIN_IDS)}")
print(f"Val cases  : {len(VAL_IDS)}")
print(f"Val IDs    : {VAL_IDS}")

## Dataset Conversion (Segmentation only, .nii.gz)


In [ ]:
DATASET_DIR   = "/kaggle/working/nnUNet_raw/Dataset003_Liver"
images_tr_dir = os.path.join(DATASET_DIR, "imagesTr")
labels_tr_dir = os.path.join(DATASET_DIR, "labelsTr")

# Clean previous conversion
shutil.rmtree(images_tr_dir, ignore_errors=True)
shutil.rmtree(labels_tr_dir, ignore_errors=True)
os.makedirs(images_tr_dir, exist_ok=True)
os.makedirs(labels_tr_dir, exist_ok=True)

print(f"Converting {len(TRAIN_IDS)} training cases...")

for case_id in tqdm(TRAIN_IDS):
    case_name = f"liver_{case_id:03d}"

    # Volume → symlink with .nii.gz extension
    src_vol = vol_map[case_id]
    dst_vol = os.path.join(images_tr_dir, f"{case_name}_0000.nii.gz")
    if not os.path.exists(dst_vol):
        os.symlink(src_vol, dst_vol)

    # Segmentation → copy with volume header, save as .nii.gz
    vol_img  = nib.load(src_vol)
    seg_img  = nib.load(seg_map[case_id])
    seg_data = seg_img.get_fdata().astype(np.uint8)
    nib.save(
        nib.Nifti1Image(seg_data, vol_img.affine, vol_img.header),
        os.path.join(labels_tr_dir, f"{case_name}.nii.gz")
    )

print(f"\nimagesTr: {len(os.listdir(images_tr_dir))} files")
print(f"labelsTr: {len(os.listdir(labels_tr_dir))} files")

total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Used: {used / 1e9:.1f} GB")
print(f"Free: {free / 1e9:.1f} GB")

## dataset.json


In [ ]:
# Volume files are .nii, segmentation files are .nii.gz
# nnU-Net uses file_ending for imagesTr — set to .nii
dataset_json = {
    "channel_names": {"0": "CT"},
    "labels": {
        "background": 0,
        "liver": 1,
        "tumor": 2
    },
    "numTraining": len(TRAIN_IDS),
    "file_ending": ".nii.gz"
}

json_path = os.path.join(DATASET_DIR, "dataset.json")
with open(json_path, "w") as f:
    json.dump(dataset_json, f, indent=2)

print("dataset.json created:")
print(json.dumps(dataset_json, indent=2))

## Preprocessing

In [ ]:
result = subprocess.run([
    "nnUNetv2_plan_and_preprocess",
    "-d", "3",
    "-c", "2d",
    "-np", "2"
], capture_output=True, text=True, env=os.environ)

print("=== STDOUT (last 2000 chars) ===")
print(result.stdout[-2000:])
print("=== STDERR (last 500 chars) ===")
print(result.stderr[-500:] if result.stderr else "No errors")
print("=== Return code ===", result.returncode)

# Check disk after preprocessing
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"\nDisk after preprocessing:")
print(f"Used: {used / 1e9:.1f} GB")
print(f"Free: {free / 1e9:.1f} GB")

## Clean Up Raw Data


In [ ]:
# Delete nnUNet_raw — no longer needed
shutil.rmtree("/kaggle/working/nnUNet_raw")

total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Used: {used / 1e9:.1f} GB")
print(f"Free: {free / 1e9:.1f} GB")
print("Ready for training.")

## Environment Variables (Session Başında Tekrar Set Et)


In [ ]:
import os, shutil, subprocess

os.environ["nnUNet_raw"]          = "/kaggle/working/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/kaggle/working/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = "/kaggle/working/nnUNet_results"
os.environ["nnUNet_n_proc_DA"]    = "2"

# check preprocessing
pp = "/kaggle/working/nnUNet_preprocessed/Dataset003_Liver"
print("Preprocessed exists:", os.path.isdir(pp))
print("Contents:", os.listdir(pp))

total, used, free = shutil.disk_usage("/kaggle/working")
print(f"\nUsed: {used/1e9:.1f} GB  |  Free: {free/1e9:.1f} GB")

## Eğitim (Fold 0, 2D)

In [ ]:
result = subprocess.run([
    "nnUNetv2_train",
    "3",
    "2d",
    "0",
    "--npz",
    "-tr", "nnUNetTrainer_250epochs",
], env=os.environ)

print("Return code:", result.returncode)

In [ ]:
import shutil, os

shutil.copytree(
    "/kaggle/working/nnUNet_results",
    "/kaggle/working/nnUNet_results_backup"
)
print("Checkpoint kaydedildi.")

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/working/nnUNet_results"):
    for f in files:
        path = os.path.join(root, f)
        print(f"{path}  {os.path.getsize(path)/1e6:.0f} MB")

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/working/nnUNet_results"):
    for f in files:
        path = os.path.join(root, f)
        print(f"{path}  {os.path.getsize(path)/1e6:.0f} MB")

In [ ]:
import shutil
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Used: {used/1e9:.1f} GB  |  Free: {free/1e9:.1f} GB")

In [ ]:
import shutil, os

src = "/kaggle/working/nnUNet_results/Dataset003_Liver/nnUNetTrainer_250epochs__nnUNetPlans__2d"
dst = "/kaggle/working/nnUNet_results_SAVE/Dataset003_Liver/nnUNetTrainer_250epochs__nnUNetPlans__2d"

shutil.copytree(src, dst, dirs_exist_ok=True)
print("Kaydedildi.")

In [ ]:
log_path = "/kaggle/working/nnUNet_results/Dataset003_Liver/nnUNetTrainer_250epochs__nnUNetPlans__2d/fold_0/training_log_2026_4_19_22_08_45.txt"

with open(log_path) as f:
    lines = f.readlines()

# Sadece en iyi EMA satırlarını göster
best_lines = [l for l in lines if "New best EMA" in l]
print(f"Toplam {len(best_lines)} kez yeni best:")
print(best_lines[-5:])  # Son 5 best

In [ ]:
log_path = "/kaggle/working/nnUNet_results/Dataset003_Liver/nnUNetTrainer_250epochs__nnUNetPlans__2d/fold_0/training_log_2026_4_19_22_08_45.txt"

with open(log_path) as f:
    lines = f.readlines()

# Pseudo dice satırlarını göster - son 10 epoch
pseudo_lines = [l for l in lines if "Pseudo dice" in l]
print(f"Toplam epoch: {len(pseudo_lines)}")
print("\nSon 10 epoch:")
for l in pseudo_lines[-10:]:
    print(l.strip())

### is there any overfitting ?

In [ ]:
train_lines = [l for l in lines if "train_loss" in l]
val_lines = [l for l in lines if "val_loss" in l]

print("Son 10 epoch train vs val loss:")
for t, v in zip(train_lines[-10:], val_lines[-10:]):
    t_val = t.strip().split()[-1]
    v_val = v.strip().split()[-1]
    print(f"train: {t_val}  |  val: {v_val}")

## Let's create the necessary folder structure for nnUNet:

In [ ]:
import shutil, os

# nnUNet results klasor yapisi
results_dir = "/kaggle/working/nnUNet_results/Dataset003_Liver/nnUNetTrainer_250epochs__nnUNetPlans__2d/fold_0"
os.makedirs(results_dir, exist_ok=True)

# checkpoint kopyala
shutil.copy("/kaggle/input/datasets/eminmanov/nnunet-liver-checkpoint/checkpoint_best.pth", results_dir)
shutil.copy("/kaggle/input/datasets/eminmanov/nnunet-liver-checkpoint/checkpoint_latest.pth", results_dir)
shutil.copy("/kaggle/input/datasets/eminmanov/nnunet-liver-checkpoint/plans.json",
            "/kaggle/working/nnUNet_results/Dataset003_Liver/nnUNetTrainer_250epochs__nnUNetPlans__2d/")
shutil.copy("/kaggle/input/datasets/eminmanov/nnunet-liver-checkpoint/dataset.json",
            "/kaggle/working/nnUNet_results/Dataset003_Liver/nnUNetTrainer_250epochs__nnUNetPlans__2d/")
shutil.copy("/kaggle/input/datasets/eminmanov/nnunet-liver-checkpoint/dataset_fingerprint.json",
            "/kaggle/working/nnUNet_results/Dataset003_Liver/nnUNetTrainer_250epochs__nnUNetPlans__2d/")

print("Done.")
print(os.listdir(results_dir))

## Let's prepare the volume files for the 20 validation cases:

In [ ]:
import os, shutil

val_input_dir = "/kaggle/working/val_images"
os.makedirs(val_input_dir, exist_ok=True)

for case_id in VAL_IDS:
    src = vol_map[case_id]
    dst = os.path.join(val_input_dir, f"liver_{case_id:03d}_0000.nii.gz")
    if src.endswith(".nii.gz"):
        shutil.copy(src, dst)
    else:
        import nibabel as nib
        import numpy as np
        img = nib.load(src)
        nib.save(img, dst)

print(f"Val images: {len(os.listdir(val_input_dir))}")
print(os.listdir(val_input_dir))

## 20 cases are ready. Now let's run the prediction:

In [ ]:
result = subprocess.run([
    "nnUNetv2_predict",
    "-i", "/kaggle/working/val_images",
    "-o", "/kaggle/working/val_preds",
    "-d", "3",
    "-c", "2d",
    "-f", "0",
    "-tr", "nnUNetTrainer_250epochs",
    "-chk", "checkpoint_best.pth",
    "-npp", "1",
    "-nps", "1",
], env=os.environ)

print("Return code:", result.returncode)

### Let's calculate case-level Dice:

In [ ]:
import nibabel as nib
import numpy as np
import os

def dice_score(pred, gt, label):
    pred_mask = (pred == label)
    gt_mask = (gt == label)
    intersection = (pred_mask & gt_mask).sum()
    total = pred_mask.sum() + gt_mask.sum()
    if total == 0:
        return 1.0
    return 2.0 * intersection / total

pred_dir = "/kaggle/working/val_preds"
results = []

for case_id in VAL_IDS:
    case_name = f"liver_{case_id:03d}"
    pred_path = os.path.join(pred_dir, f"{case_name}.nii.gz")
    gt_path = seg_map[case_id]

    pred = nib.load(pred_path).get_fdata().astype(np.uint8)
    gt = nib.load(gt_path).get_fdata().astype(np.uint8)

    liver_dice = dice_score(pred, gt, 1)
    tumor_dice = dice_score(pred, gt, 2)
    results.append((case_id, liver_dice, tumor_dice))
    print(f"Case {case_id:03d} | Liver: {liver_dice:.4f} | Tumor: {tumor_dice:.4f}")

liver_mean = np.mean([r[1] for r in results])
tumor_mean = np.mean([r[2] for r in results])
print(f"\nMean Liver Dice: {liver_mean:.4f}")
print(f"Mean Tumor Dice: {tumor_mean:.4f}")

### 1. Training Curve (Loss ve Pseudo Dice) - Visualization


In [ ]:
import matplotlib.pyplot as plt
import re

log_path = "/kaggle/working/nnUNet_results/Dataset003_Liver/nnUNetTrainer_250epochs__nnUNetPlans__2d/fold_0/training_log_2026_4_19_22_08_45.txt"

with open(log_path) as f:
    lines = f.readlines()

# Pseudo dice değerlerini çek
pseudo_dice = []
for l in lines:
    if "Pseudo dice" in l:
        nums = re.findall(r'\d+\.\d+', l)
        if nums:
            pseudo_dice.append(float(nums[-1]))

# Train loss değerlerini çek
train_losses = []
for l in lines:
    if "train_loss" in l:
        nums = re.findall(r'-?\d+\.\d+', l)
        if nums:
            train_losses.append(float(nums[-1]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, color='steelblue', linewidth=1.5)
ax1.set_title('Training Loss', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(pseudo_dice, color='darkorange', linewidth=1.5)
ax2.set_title('Validation Pseudo Dice', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dice Score')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 2. Case-Level Dice Bar Chart - Visualization


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

case_ids  = [r[0] for r in results]
liver_d   = [r[1] for r in results]
tumor_d   = [r[2] for r in results]

x = np.arange(len(case_ids))
width = 0.35

fig, ax = plt.subplots(figsize=(16, 6))
bars1 = ax.bar(x - width/2, liver_d, width, label='Liver Dice', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, tumor_d, width, label='Tumor Dice', color='tomato', alpha=0.85)

ax.axhline(np.mean(liver_d), color='steelblue', linestyle='--', linewidth=1.5,
           label=f'Mean Liver: {np.mean(liver_d):.4f}')
ax.axhline(np.mean(tumor_d), color='tomato', linestyle='--', linewidth=1.5,
           label=f'Mean Tumor: {np.mean(tumor_d):.4f}')

ax.set_xlabel('Case ID', fontsize=12)
ax.set_ylabel('Dice Score', fontsize=12)
ax.set_title('Case-Level Dice Scores — nnU-Net 2D (250 epochs)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([f'Case\n{c}' for c in case_ids], fontsize=8)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig('case_dice_scores.png', dpi=150, bbox_inches='tight')
plt.show()

### 3. CT Image + Prediction Overlay

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import nibabel as nib

def visualize_case(case_id, slice_offset=0):
    vol  = nib.load(vol_map[case_id]).get_fdata()
    gt   = nib.load(seg_map[case_id]).get_fdata().astype(np.uint8)
    pred = nib.load(f"/kaggle/working/val_preds/liver_{case_id:03d}.nii.gz").get_fdata().astype(np.uint8)

    # En iyi slice'ı bul (en fazla tümör içeren)
    tumor_per_slice = [(gt[:, :, z] == 2).sum() for z in range(gt.shape[2])]
    best_z = np.argmax(tumor_per_slice) + slice_offset
    best_z = np.clip(best_z, 0, gt.shape[2] - 1)

    ct_slice   = vol[:, :, best_z]
    gt_slice   = gt[:, :, best_z]
    pred_slice = pred[:, :, best_z]

    # HU windowing for display
    ct_display = np.clip(ct_slice, -100, 400)
    ct_display = (ct_display - ct_display.min()) / (ct_display.max() - ct_display.min())

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f'Case {case_id:03d} — Slice {best_z}', fontsize=14)

    # CT only
    axes[0].imshow(ct_display, cmap='gray')
    axes[0].set_title('CT Image', fontsize=12)
    axes[0].axis('off')

    # CT + Ground Truth
    axes[1].imshow(ct_display, cmap='gray')
    overlay_gt = np.zeros((*ct_display.shape, 4))
    overlay_gt[gt_slice == 1] = [0, 0, 1, 0.3]   # liver: blue
    overlay_gt[gt_slice == 2] = [1, 0, 0, 0.6]   # tumor: red
    axes[1].imshow(overlay_gt)
    axes[1].set_title('Ground Truth', fontsize=12)
    axes[1].axis('off')

    # CT + Prediction
    axes[2].imshow(ct_display, cmap='gray')
    overlay_pred = np.zeros((*ct_display.shape, 4))
    overlay_pred[pred_slice == 1] = [0, 0, 1, 0.3]
    overlay_pred[pred_slice == 2] = [1, 0, 0, 0.6]
    axes[2].imshow(overlay_pred)
    axes[2].set_title('Prediction', fontsize=12)
    axes[2].axis('off')

    liver_patch = mpatches.Patch(color='blue', alpha=0.5, label='Liver')
    tumor_patch = mpatches.Patch(color='red', alpha=0.6, label='Tumor')
    fig.legend(handles=[liver_patch, tumor_patch], loc='lower center',
               ncol=2, fontsize=11, bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout()
    plt.savefig(f'case_{case_id:03d}_overlay.png', dpi=150, bbox_inches='tight')
    plt.show()

# İyi ve kötü örnekleri göster
for cid in [114, 69, 17, 54, 86]:  # iyi → kötü
    visualize_case(cid)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
import nibabel as nib
import os

def visualize_case_detailed(case_id, pred_dir, vol_map, seg_map):
    
    vol  = nib.load(vol_map[case_id]).get_fdata()
    gt   = nib.load(seg_map[case_id]).get_fdata().astype(np.uint8)
    pred = nib.load(os.path.join(pred_dir, f"liver_{case_id:03d}.nii.gz")).get_fdata().astype(np.uint8)

    # Case-level Dice hesapla
    def dice(p, g, label):
        pm = (p == label); gm = (g == label)
        inter = (pm & gm).sum(); total = pm.sum() + gm.sum()
        return 1.0 if total == 0 else 2.0 * inter / total

    liver_dice = dice(pred, gt, 1)
    tumor_dice = dice(pred, gt, 2)

    # En fazla tümör içeren slice'ı bul
    tumor_per_slice = [(gt[:,:,z] == 2).sum() for z in range(gt.shape[2])]
    best_z = int(np.argmax(tumor_per_slice))

    ct_slice   = vol[:,:,best_z]
    gt_slice   = gt[:,:,best_z]
    pred_slice = pred[:,:,best_z]

    # HU windowing
    ct_disp = np.clip(ct_slice, -100, 400)
    ct_disp = (ct_disp - ct_disp.min()) / (ct_disp.max() - ct_disp.min() + 1e-8)

    # Overlay oluştur
    def make_overlay(ct, gt_s, pred_s):
        rgb = np.stack([ct, ct, ct], axis=-1)
        overlay = np.zeros((*ct.shape, 4), dtype=np.float32)
        # Liver: mavi
        overlay[gt_s == 1]   = [0.2, 0.4, 0.9, 0.30]
        # Tumor GT: kırmızı
        overlay[gt_s == 2]   = [0.9, 0.1, 0.1, 0.65]
        # Tumor Pred: yeşil
        overlay[pred_s == 2] = [0.1, 0.85, 0.3, 0.65]
        # Overlap (TP): sarı
        tp = (gt_s == 2) & (pred_s == 2)
        overlay[tp] = [1.0, 0.85, 0.0, 0.70]
        return rgb, overlay

    rgb, overlay = make_overlay(ct_disp, gt_slice, pred_slice)

    # Renk performans sınıfı
    if tumor_dice >= 0.80:
        perf_color = '#2ecc71'; perf_label = 'Excellent'
    elif tumor_dice >= 0.60:
        perf_color = '#f39c12'; perf_label = 'Good'
    elif tumor_dice >= 0.30:
        perf_color = '#e67e22'; perf_label = 'Fair'
    else:
        perf_color = '#e74c3c'; perf_label = 'Poor'

    # Figure
    fig = plt.figure(figsize=(16, 7), facecolor='#0d1117')
    gs  = gridspec.GridSpec(2, 4, figure=fig,
                             height_ratios=[1, 0.08],
                             hspace=0.12, wspace=0.06)

    ax_ct   = fig.add_subplot(gs[0, 0])
    ax_gt   = fig.add_subplot(gs[0, 1])
    ax_pred = fig.add_subplot(gs[0, 2])
    ax_diff = fig.add_subplot(gs[0, 3])

    # 1 — CT
    ax_ct.imshow(ct_disp.T, cmap='gray', origin='lower')
    ax_ct.set_title('CT Image', color='white', fontsize=12, pad=6)
    ax_ct.text(0.02, 0.98, f'Slice {best_z}', transform=ax_ct.transAxes,
               color='#aaaaaa', fontsize=9, va='top')

    # 2 — Ground Truth overlay
    ax_gt.imshow(ct_disp.T, cmap='gray', origin='lower')
    ov_gt = np.zeros((*ct_disp.T.shape, 4))
    ov_gt[gt_slice.T == 1] = [0.2, 0.4, 0.9, 0.35]
    ov_gt[gt_slice.T == 2] = [0.9, 0.1, 0.1, 0.70]
    ax_gt.imshow(ov_gt, origin='lower')
    ax_gt.set_title('Ground Truth', color='white', fontsize=12, pad=6)

    # 3 — Prediction overlay
    ax_pred.imshow(ct_disp.T, cmap='gray', origin='lower')
    ov_pred = np.zeros((*ct_disp.T.shape, 4))
    ov_pred[pred_slice.T == 1] = [0.2, 0.4, 0.9, 0.35]
    ov_pred[pred_slice.T == 2] = [0.1, 0.85, 0.3, 0.70]
    ax_pred.imshow(ov_pred, origin='lower')
    ax_pred.set_title('Prediction', color='white', fontsize=12, pad=6)

    # 4 — Fark haritası (TP / FP / FN)
    ax_diff.imshow(ct_disp.T, cmap='gray', origin='lower')
    diff = np.zeros((*ct_disp.T.shape, 4))
    gt_t   = gt_slice.T == 2
    pred_t = pred_slice.T == 2
    diff[gt_t & pred_t]   = [1.0, 0.85, 0.0, 0.80]  # TP: sarı
    diff[~gt_t & pred_t]  = [0.1, 0.85, 0.3, 0.80]  # FP: yeşil
    diff[gt_t & ~pred_t]  = [0.9, 0.1, 0.1, 0.80]   # FN: kırmızı
    ax_diff.imshow(diff, origin='lower')
    ax_diff.set_title('Error Map', color='white', fontsize=12, pad=6)

    for ax in [ax_ct, ax_gt, ax_pred, ax_diff]:
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_edgecolor('#333333')

    # Alt şerit — metrikler
    ax_bar = fig.add_subplot(gs[1, :])
    ax_bar.axis('off')

    # Liver dice bar
    fig.text(0.09, 0.10, f'Liver Dice: {liver_dice:.4f}',
             color='#5599ff', fontsize=11, ha='center', va='center',
             fontweight='bold')
    fig.text(0.30, 0.10, f'Tumor Dice: {tumor_dice:.4f}',
             color=perf_color, fontsize=11, ha='center', va='center',
             fontweight='bold')
    fig.text(0.52, 0.10, f'Performance: {perf_label}',
             color=perf_color, fontsize=11, ha='center', va='center',
             fontweight='bold')
    fig.text(0.72, 0.10, f'Case {case_id:03d}',
             color='#aaaaaa', fontsize=11, ha='center', va='center')

    # Legend
    legend_patches = [
        mpatches.Patch(color='#3366ee', alpha=0.6, label='Liver'),
        mpatches.Patch(color='#dd2222', alpha=0.7, label='Tumor GT'),
        mpatches.Patch(color='#22cc55', alpha=0.7, label='Tumor Pred'),
        mpatches.Patch(color='#ffcc00', alpha=0.8, label='TP (overlap)'),
    ]
    fig.legend(handles=legend_patches, loc='lower right',
               ncol=4, fontsize=9,
               facecolor='#1a1a2e', edgecolor='#333333',
               labelcolor='white',
               bbox_to_anchor=(0.99, 0.01))

    fig.suptitle(f'nnU-Net 2D — LiTS Segmentation · Case {case_id:03d}',
                 color='white', fontsize=14, y=0.98)

    plt.savefig(f'overlay_case_{case_id:03d}.png',
                dpi=150, bbox_inches='tight',
                facecolor='#0d1117')
    plt.show()
    plt.close()


# Tüm val case'leri çalıştır
pred_dir = "/kaggle/working/val_preds"

for case_id in VAL_IDS:
    visualize_case_detailed(case_id, pred_dir, vol_map, seg_map)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
import nibabel as nib
import os

def visualize_case(case_id, pred_dir, vol_map, seg_map):
    
    vol  = nib.load(vol_map[case_id]).get_fdata()
    gt   = nib.load(seg_map[case_id]).get_fdata().astype(np.uint8)
    pred = nib.load(os.path.join(pred_dir, f"liver_{case_id:03d}.nii.gz")).get_fdata().astype(np.uint8)

    def dice(p, g, label):
        pm = (p == label); gm = (g == label)
        inter = (pm & gm).sum(); total = pm.sum() + gm.sum()
        return 1.0 if total == 0 else 2.0 * inter / total

    liver_dice = dice(pred, gt, 1)
    tumor_dice = dice(pred, gt, 2)

    # Best slice — most tumor
    tumor_per_slice = [(gt[:,:,z] == 2).sum() for z in range(gt.shape[2])]
    best_z = int(np.argmax(tumor_per_slice))

    ct_slice   = vol[:,:,best_z]
    gt_slice   = gt[:,:,best_z]
    pred_slice = pred[:,:,best_z]

    ct_disp = np.clip(ct_slice, -100, 400)
    ct_disp = (ct_disp - ct_disp.min()) / (ct_disp.max() - ct_disp.min() + 1e-8)

    fig, axes = plt.subplots(1, 3, figsize=(13, 5), facecolor='#0d1117')
    fig.suptitle(f'nnU-Net 2D — Case {case_id:03d}  |  Liver Dice: {liver_dice:.4f}  |  Tumor Dice: {tumor_dice:.4f}',
                 color='white', fontsize=13, y=1.01)

    # Panel 1 — CT only
    axes[0].imshow(ct_disp.T, cmap='gray', origin='lower')
    axes[0].set_title('CT Image', color='white', fontsize=11, pad=6)
    axes[0].text(0.02, 0.98, f'Slice {best_z}', transform=axes[0].transAxes,
                 color='#aaaaaa', fontsize=8, va='top')

    # Panel 2 — Ground Truth
    axes[1].imshow(ct_disp.T, cmap='gray', origin='lower')
    ov_gt = np.zeros((*ct_disp.T.shape, 4))
    ov_gt[gt_slice.T == 1] = [0.2, 0.4, 0.9, 0.35]
    ov_gt[gt_slice.T == 2] = [0.9, 0.1, 0.1, 0.70]
    axes[1].imshow(ov_gt, origin='lower')
    axes[1].set_title('Ground Truth', color='white', fontsize=11, pad=6)

    # Panel 3 — Prediction
    axes[2].imshow(ct_disp.T, cmap='gray', origin='lower')
    ov_pred = np.zeros((*ct_disp.T.shape, 4))
    ov_pred[pred_slice.T == 1] = [0.2, 0.4, 0.9, 0.35]
    ov_pred[pred_slice.T == 2] = [0.1, 0.85, 0.3, 0.70]
    axes[2].imshow(ov_pred, origin='lower')
    axes[2].set_title('Prediction', color='white', fontsize=11, pad=6)

    for ax in axes:
        ax.axis('off')

    # Legend
    legend_patches = [
        mpatches.Patch(color='#3366ee', alpha=0.6, label='Liver'),
        mpatches.Patch(color='#dd2222', alpha=0.7, label='Tumor (GT)'),
        mpatches.Patch(color='#22cc55', alpha=0.7, label='Tumor (Pred)'),
    ]
    fig.legend(handles=legend_patches, loc='lower center',
               ncol=3, fontsize=10,
               facecolor='#1a1a2e', edgecolor='#333333',
               labelcolor='white',
               bbox_to_anchor=(0.5, -0.04))

    plt.tight_layout()
    plt.savefig(f'overlay_case_{case_id:03d}.png',
                dpi=150, bbox_inches='tight',
                facecolor='#0d1117')
    plt.show()
    plt.close()


# Run for all val cases
pred_dir = "/kaggle/working/val_preds"
for case_id in VAL_IDS:
    visualize_case(case_id, pred_dir, vol_map, seg_map)

## Summary Metrics Table

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics = {
    'Model': ['nnU-Net 2D\n(Ours)', 'Attention U-Net\n(Ours)', 'SMP ResNet34\n(Ours)'],
    'Liver Dice': [0.9476, 0.929, 0.95],   # attention ve smp'yi kendi sonuçlarınla güncelle
    'Tumor Dice': [0.6241, 0.517, 0.691],
}

x = np.arange(len(metrics['Model']))
width = 0.3

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width/2, metrics['Liver Dice'], width, label='Liver Dice',
       color='steelblue', alpha=0.85)
ax.bar(x + width/2, metrics['Tumor Dice'], width, label='Tumor Dice',
       color='tomato', alpha=0.85)

ax.set_ylabel('Dice Score', fontsize=12)
ax.set_title('Model Comparison — LiTS Dataset', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(metrics['Model'], fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.2, axis='y')

for i, (l, t) in enumerate(zip(metrics['Liver Dice'], metrics['Tumor Dice'])):
    ax.text(i - width/2, l + 0.01, f'{l:.3f}', ha='center', fontsize=10)
    ax.text(i + width/2, t + 0.01, f'{t:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### TP/FP/FN breakdown bar chart

In [ ]:
results_detail = []

for case_id in VAL_IDS:
    gt   = nib.load(seg_map[case_id]).get_fdata().astype(np.uint8)
    pred = nib.load(f"/kaggle/working/val_preds/liver_{case_id:03d}.nii.gz").get_fdata().astype(np.uint8)
    
    gt_t   = (gt == 2)
    pred_t = (pred == 2)
    
    tp = (gt_t & pred_t).sum()
    fp = (~gt_t & pred_t).sum()
    fn = (gt_t & ~pred_t).sum()
    
    results_detail.append({'case': case_id, 'TP': tp, 'FP': fp, 'FN': fn})

cases  = [r['case'] for r in results_detail]
tps    = [r['TP'] for r in results_detail]
fps    = [r['FP'] for r in results_detail]
fns    = [r['FN'] for r in results_detail]

x = np.arange(len(cases))
fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(x, tps, label='TP', color='#2ecc71', alpha=0.85)
ax.bar(x, fps, bottom=tps, label='FP', color='#e74c3c', alpha=0.85)
ax.bar(x, fns, bottom=np.array(tps)+np.array(fps), label='FN', color='#f39c12', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f'Case\n{c}' for c in cases], fontsize=8)
ax.set_ylabel('Voxel count')
ax.set_title('Tumor Segmentation — TP / FP / FN Breakdown per Case')
ax.legend()
plt.tight_layout()
plt.savefig('tp_fp_fn_breakdown.png', dpi=150)
plt.show()